# LAB 09 — Tokenization và label alignment


In [1]:
import sys, json, inspect
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import display
LAB_DIR = Path.cwd() / 'lab9' if (Path.cwd() / 'lab9/baseline_ner.py').is_file() else Path.cwd()
sys.path.insert(0, str(LAB_DIR.resolve()))
import baseline_ner as ner
ner.configure()
model_path = ner.assets()
tokenizer = ner.BertTokenizerFast.from_pretrained(model_path, do_lower_case=True)
data = json.loads((ner.CACHE/'data.json').read_text(encoding='utf-8'))
print({split: len(rows) for split, rows in data.items()})

{'train': 14041, 'validation': 3250, 'test': 3453}


W1112 07:27:15.576000 5308 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


#### 3. Dataset

CoNLL-2003 English, nguồn `tomaarsen/conll2003`; train/validation/test = 14041/3250/3453 câu. Nhãn PER/ORG/LOC/MISC và O. Giữ split gốc, không chọn model bằng test. Dữ liệu được chuẩn hóa BIO2: I-X đầu chunk đổi thành B-X mà không thay entity spans.


#### 4. Sequence Classification vs Token Classification

Sequence classification: H → H[:,0] → head → (B,C), một nhãn mỗi câu. Token classification: H → cùng một Linear ở mọi vị trí → (B,T,C), một nhãn mỗi token được giám sát.


#### 5. BIO Tagging

B-X bắt đầu entity X; I-X tiếp tục; O ngoài entity. `Barack/B-PER Obama/I-PER visited/O New/B-LOC York/I-LOC ./O`.


#### 8. Subword Tokenization

Một word có thể tạo nhiều WordPiece. Vocabulary uncased giữ được nhiều tên qua subword nhưng bỏ phân biệt hoa/thường. Nhãn gốc theo word phải được nối với token bằng word_ids; không tự đoán cách tokenizer tách tên.


#### 9. Label Alignment

Original words → word-level labels → tokenizer → subword tokens → word_ids → aligned token labels. Special tokens có word_id=None.


#### 10. Alignment Strategies

Chọn Strategy A: chỉ subword đầu nhận label; phần tiếp nối và special tokens nhận -100. Cách này giữ mỗi word có một đóng góp loss và một dự đoán khi đánh giá. Mảnh tiếp nối vẫn tham gia self-attention và ảnh hưởng biểu diễn token đầu; không có loss trực tiếp riêng. Không lan B-X sang mọi mảnh để tránh tạo entity giả.


In [2]:
example = {'words': ['Elon','Musk','founded','SpaceX','in','California','.'],
           'tags': ['B-PER','I-PER','O','B-ORG','O','B-LOC','O']}
encoded, labels = ner.align(tokenizer, example['words'], example['tags'])
display(pd.DataFrame({'tokens': tokenizer.convert_ids_to_tokens(encoded['input_ids']),
 'input_ids': encoded['input_ids'], 'attention_mask': encoded['attention_mask'],
 'word_ids': encoded.word_ids(), 'labels': labels,
 'BIO': [ner.TAGS[x] if x != -100 else 'IGNORE' for x in labels]}))

,tokens,input_ids,attention_mask,word_ids,labels,BIO
0,[CLS],101,1,NaN,-100,IGNORE
1,el,3449,1,0.0,1,B-PER
2,##on,2239,1,0.0,-100,IGNORE
3,mu,14163,1,1.0,2,I-PER
4,##sk,6711,1,1.0,-100,IGNORE
5,founded,2631,1,2.0,0,O
6,space,2686,1,3.0,3,B-ORG
7,##x,2595,1,3.0,-100,IGNORE
8,in,1999,1,4.0,0,O
9,california,2662,1,5.0,5,B-LOC
